# Neuron Wikipedia pretraining on Google Colab

This notebook prepares a Wikipedia corpus and trains the Neuron hybrid language model. It uses the Llama 2 tokenizer by default; training a new byte-level BPE tokenizer is an explicit option. It includes setup, credentials, data preparation, a pipeline check, ablations, training, and output inspection.

The current release implements pretraining. Supervised fine-tuning (SFT) and reinforcement learning (RL) are planned parts of the upcoming end-to-end training workflow.


## 1. Choose GPU and repository

The notebook downloads the public repository from GitHub by default. Change `REPO_URL` to a fork or set it to an empty string when you have already uploaded or mounted a checkout at `REPO_DIR`.

In [ ]:
#@title Repository location
REPO_URL = "https://github.com/CreatorDevX/Neuron.git"  #@param {type:"string"}
REPO_DIR = "/content/Neuron"  #@param {type:"string"}

from pathlib import Path
import os, subprocess
if REPO_URL:
    if Path(REPO_DIR).exists():
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
if not Path(REPO_DIR, "neuron", "train.py").exists():
    raise FileNotFoundError(f"Repository not found at {REPO_DIR}; set REPO_URL or REPO_DIR")
os.chdir(REPO_DIR)
print("Repository:", Path.cwd())


In [ ]:
#@title Check the Colab accelerator
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("In Colab, select Runtime > Change runtime type > GPU, then rerun.")
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


## 2. Install dependencies

This installs the project requirements and the optional Flash Linear Attention CUDA backend for GDN. The default sparse attention implementation is included in Neuron and uses PyTorch SDPA. If Colab requests a runtime restart, restart and continue from the accelerator check.


In [ ]:
%pip install -r requirements.txt
%pip install "flash-linear-attention[cuda]"


## Optional: build FlashAttention for Turing (T4)

Run these cells only to benchmark `attention=flash_turing`. The CUDA extension compiles for SM75 and may take several minutes. The Neuron checkout must include `third_party/flash-attention-turing/` (and its Neuron patch). If the CUTLASS submodule files are missing from that checkout, the cell downloads the pinned CUTLASS revision into the runtime. The Turing source has no upstream root license file at this time; the Neuron Apache-2.0 license does not apply to it. See `THIRD_PARTY_NOTICES.md`.


In [ ]:
#@title Build optional Turing FlashAttention extension
from pathlib import Path
import os, shutil, subprocess, sys

turing_dir = Path("third_party/flash-attention-turing")
if not (turing_dir / "flash_attention_interface.py").exists():
    raise FileNotFoundError("The Neuron checkout must include third_party/flash-attention-turing")
cutlass_dir = turing_dir / "csrc/cutlass"
cutlass_header = cutlass_dir / "include/cutlass/cutlass.h"
if not cutlass_header.exists():
    staging = Path("/content/cutlass-build")
    if staging.exists():
        shutil.rmtree(staging)
    subprocess.run(["git", "clone", "https://github.com/NVIDIA/cutlass.git", str(staging)], check=True)
    subprocess.run(["git", "-C", str(staging), "checkout", "df18f5e4f5de76bed8be1de8e4c245f2f5ec3020"], check=True)
    stale_git = cutlass_dir / ".git"
    if stale_git.is_file():
        stale_git.unlink()
    elif stale_git.is_dir():
        shutil.rmtree(stale_git)
    shutil.copytree(staging, cutlass_dir, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".git"))
build_env = dict(os.environ, MAX_JOBS="2")
subprocess.run([sys.executable, "-m", "pip", "install", "-v", "."], cwd=turing_dir, env=build_env, check=True)
print("Turing FlashAttention extension built.")


## 3. Configure credentials

Choose one of the next two cells. If you created Google Colab Secrets, run the first cell. Otherwise skip it and fill in the manual values in the second cell. The notebook writes `.env` locally and does not display the values.

| Name | What it is used for | Required? |
|---|---|---|
| `HF_TOKEN` | Hugging Face user access token. Needs read access to the gated Llama 2 tokenizer. Add write access if you will upload files to a Hugging Face Storage Bucket. | Required for Llama 2 download; otherwise optional |
| `HF_USERNAME` | Your Hugging Face account username; used to find your bucket when no namespace is given. | Optional |
| `HF_NAMESPACE` | Hugging Face account or organization that owns the bucket. | Optional |
| `HF_BUCKET_NAME` | Name of an existing Hugging Face Storage Bucket. | Optional |
| `HF_BUCKET_ID` | Full bucket id, usually `owner/name`. Use this instead of username/namespace plus bucket name. | Optional |
| `HF_BUCKET_PREFIX` | Folder prefix inside the bucket for data, tokenizer, checkpoints, and logs. | Optional; defaults to `neuron-wikipedia` |
| `WANDB_API_KEY` | API key from your Weights & Biases account settings. | Optional |
| `WANDB_USERNAME` | W&B account username. | Optional |
| `WANDB_ENTITY` | W&B user or team that owns the project. | Optional |
| `WANDB_PROJECT` | W&B project slug where run metrics are logged. | Optional; defaults to `neuron-wikipedia-pretrain` |
| `WANDB_RUN_ID` | Existing run id to resume tracking. Usually leave blank for automatic creation. | Optional |
| `WANDB_MODE` | Logging mode: `online`, `offline`, or `disabled`. | Optional; defaults to `online` |

The Wikipedia dataset is public. Meta gates access to its Llama 2 tokenizer: accept Meta's terms and request Hugging Face access before downloading it.


In [ ]:
#@title Optional: load matching values from Google Colab Secrets
from pathlib import Path

secret_names = [
    "HF_TOKEN", "HF_USERNAME", "HF_NAMESPACE", "HF_BUCKET_NAME", "HF_BUCKET_ID",
    "HF_BUCKET_PREFIX", "WANDB_API_KEY", "WANDB_USERNAME", "WANDB_ENTITY",
    "WANDB_PROJECT", "WANDB_RUN_ID", "WANDB_MODE",
]
defaults = {"HF_BUCKET_PREFIX": "neuron-wikipedia", "WANDB_PROJECT": "neuron-wikipedia-pretrain", "WANDB_MODE": "online"}
values = {}
try:
    from google.colab import userdata
except ImportError:
    print("Colab Secrets are unavailable here; skip this cell and use the manual cell below.")
else:
    for name in secret_names:
        try:
            values[name] = userdata.get(name) or defaults.get(name, "")
        except Exception:
            values[name] = defaults.get(name, "")
    Path(".env").write_text("".join(f"{name}={values.get(name, defaults.get(name, ''))}\n" for name in secret_names), encoding="utf-8")
    print("Wrote .env from available Colab Secrets. Values hidden.")


### Manual `.env` fallback

If you do not use Colab Secrets, skip the preceding cell, set `USE_MANUAL_CREDENTIALS = True` in the code cell below, fill in the values, and run it. The default is false so a Run all does not overwrite values loaded from Colab Secrets. Keep quotation marks and only fill fields for services you use. Do not share a notebook containing real keys.


In [ ]:
#@title Manual credentials (edit values, then run this cell)
from pathlib import Path

HF_TOKEN = ""               # Hugging Face user token: read gated Llama 2; write if using bucket uploads
HF_USERNAME = ""            # Your Hugging Face account username
HF_NAMESPACE = ""           # Account or organization that owns the bucket
HF_BUCKET_NAME = ""         # Existing bucket name; optional when HF_BUCKET_ID is set
HF_BUCKET_ID = ""           # Full id such as account-or-org/bucket-name
HF_BUCKET_PREFIX = "neuron-wikipedia"  # Folder prefix in the bucket
WANDB_API_KEY = ""          # API key from your W&B account settings
WANDB_USERNAME = ""         # Your W&B username
WANDB_ENTITY = ""           # W&B user or team name
WANDB_PROJECT = "neuron-wikipedia-pretrain"  # W&B project slug
WANDB_RUN_ID = ""            # Leave blank to let training create one
WANDB_MODE = "online"       # online, offline, or disabled
USE_MANUAL_CREDENTIALS = False  #@param {type:"boolean"}

manual_values = locals()
manual_keys = [
    "HF_TOKEN", "HF_USERNAME", "HF_NAMESPACE", "HF_BUCKET_NAME", "HF_BUCKET_ID",
    "HF_BUCKET_PREFIX", "WANDB_API_KEY", "WANDB_USERNAME", "WANDB_ENTITY",
    "WANDB_PROJECT", "WANDB_RUN_ID", "WANDB_MODE",
]
if USE_MANUAL_CREDENTIALS:
    Path(".env").write_text("".join(f"{key}={manual_values[key]}\n" for key in manual_keys), encoding="utf-8")
    print("Wrote .env (contents hidden).")
else:
    print("Manual credentials skipped. Set USE_MANUAL_CREDENTIALS=True to use these values.")


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(".env", override=True)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


## 4. Confirm the active recipe

In [ ]:
from pathlib import Path
print(Path("configs/neuron-100m-t4.yaml").read_text())
print("\nData spec:\n", Path("configs/data_spec.yaml").read_text())


## 5. Reuse Llama 2 tokenizer and prepare Wikipedia shards

The default copies the fast tokenizer from `meta-llama/Llama-2-7b-hf`; it does not download model weights. This repository is gated, so accept Meta's terms and confirm your Hugging Face account has access. If you prefer a project-trained 32k byte-level BPE tokenizer, set `TRAIN_NEW_TOKENIZER = True` below. Run this setup only once per fresh data directory.

Preparation streams the configured English Wikipedia dataset without character or token truncation, creates a validation split, and writes resumable uint16 shards plus a manifest.


In [ ]:
#@title Tokenizer choice and data preparation
import subprocess, sys, yaml
from pathlib import Path

TRAIN_NEW_TOKENIZER = False  #@param {type:"boolean"}
DATA_SPEC = "configs/data_spec.yaml"
if TRAIN_NEW_TOKENIZER:
    # Keep this run's tokenizer choice explicit for auto-prepare and resume.
    spec = yaml.safe_load(Path(DATA_SPEC).read_text(encoding="utf-8"))
    spec.pop("tokenizer_repo", None)
    DATA_SPEC = "/content/neuron-data-spec.yaml"
    Path(DATA_SPEC).write_text(yaml.safe_dump(spec, sort_keys=False), encoding="utf-8")

tokenizer_cmd = [sys.executable, "-m", "neuron.train_tokenizer", "--spec", DATA_SPEC, "--out", "tokenizer"]
if TRAIN_NEW_TOKENIZER:
    tokenizer_cmd.append("--train-new")
subprocess.run(tokenizer_cmd, check=True)
subprocess.run([sys.executable, "-m", "neuron.prepare_data", "--spec", DATA_SPEC, "--out", "data"], check=True)


In [ ]:
import json
manifest = json.loads(Path("data/manifest.json").read_text())
print(json.dumps({"dataset": manifest.get("sources"), "totals": manifest.get("totals"), "vocab_size": manifest.get("vocab_size")}, indent=2))


## 6. Short real-data pipeline test

This exercises the actual loader, model, optimizer, mixed precision, and logging with a few optimizer steps. It uses batch size 1 and no compilation for a safer initial check. It does not alter the main preset.

In [ ]:
!python -m neuron.pipeline_test --config configs/neuron-100m-t4.yaml --steps 3 --set train.micro_batch=1 train.grad_accum=1 train.compile=false train.grad_checkpointing=true model.loss_chunk=128


## 7. Targeted `ablate.py` runs

Ablations use synthetic random tokens and weights, so they measure implementation speed and memory rather than model quality. Each cell writes JSONL results under `runs/`. The default commands here are deliberately small; expand `--iters`, `--warmup`, or sweep choices after confirming the runtime has enough memory.

In [ ]:
# Engram on/off, one measured iteration per case
!python -m neuron.ablate --config configs/neuron-100m-t4.yaml --device cuda --sweep engram --micro-batch 1 --iters 1 --warmup 0 --output runs/ablation-engram.jsonl


In [ ]:
# GDN backend comparison and sparse attention layout examples
!python -m neuron.ablate --config configs/neuron-100m-t4.yaml --device cuda --sweep gdn --micro-batch 1 --iters 1 --warmup 0 --output runs/ablation-gdn.jsonl
!python -m neuron.ablate --config configs/neuron-100m-t4.yaml --device cuda --sweep block_sparse --block-sizes 128 --local-windows 128 --top-blocks 0,4 --micro-batch 1 --iters 1 --warmup 0 --output runs/ablation-block-sparse.jsonl


In [ ]:
# Requires the optional extension build above and an NVIDIA T4 (SM75).
!python -m neuron.ablate --config configs/neuron-100m-t4.yaml --device cuda --sweep attention --attention flash_turing --micro-batch 1 --iters 1 --warmup 0 --output runs/ablation-flash-turing.jsonl


## 8. Start training

`profile="auto"` selects the one-GPU or two-GPU preset from the available runtime and applies its matching micro-batch and gradient-accumulation settings. Use `MANUAL_TRAIN_OVERRIDES` in the launch cell to replace any setting, for example `train.micro_batch=4` and `train.grad_accum=16`.

The selected data spec preserves the tokenizer choice during automatic preparation and resume. The run length follows the prepared training-token count. At each evaluation and checkpoint event, Neuron generates the configured prompts and appends them to `runs/neuron-pretrain/samples.jsonl`; checkpoint uploads sync that log to the Hugging Face bucket.


In [ ]:
from neuron.train import notebook_train

# Auto-select 1 GPU (Colab) or 2 GPUs (Kaggle-style profile). The profile
# supplies matching batch defaults; add typed config overrides below when needed.
MANUAL_TRAIN_OVERRIDES = []  #@param {type:"raw"}
# Example: ["train.micro_batch=4", "train.grad_accum=16"]
train_args = ["--resume", "auto", "--spec", DATA_SPEC]
if MANUAL_TRAIN_OVERRIDES:
    train_args.extend(["--set", *MANUAL_TRAIN_OVERRIDES])
notebook_train(profile="auto", args=train_args)


## 9. Inspect outputs and generate text

Training logs and checkpoints are written under `runs/neuron-100m/`. If a checkpoint exists, you can evaluate it or sample text with the commands below.

In [ ]:
!ls -lah runs/neuron-100m


In [ ]:
# Example; replace with a checkpoint filename shown above
# !python -m neuron.evaluate --checkpoint runs/neuron-100m/step_000001000.pt
# !python -m neuron.generate --checkpoint runs/neuron-100m/step_000001000.pt --prompt "The history of"


## Notes

- The Engram tables add parameters and optimizer state, so this model is about 146M parameters rather than the original 100M baseline. If memory is tight, reduce `train.micro_batch`, raise gradient accumulation to preserve effective batch size, or lower Engram table sizes.
- The block-sparse implementation uses a reference PyTorch path and is excluded from `torch.compile`; expect slower throughput than a fused sparse kernel.
- Set up a Hugging Face Storage Bucket in advance if you want remote checkpoint/data sync. Bucket credentials are optional for local Colab runs.
- The dataset configuration uses `oriental-lab/wikipedia-english`; inspect `configs/data_spec.yaml` before preparation if you want a different Wikipedia snapshot.
